In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from math import nan
from pathlib import Path
import dask
import yaml
import os
import warnings
import gc
import textwrap
warnings.filterwarnings('ignore')
import glob

In [ ]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client
dask.config.set({"distributed.scheduler.worker_saturation":1.0})

cluster = PBSCluster(
    cores = 1,
    memory = '30GB',
    processes = 1,
    queue = 'casper',
    local_directory = '$TMPDIR',
    resource_spec = 'select=1:ncpus=1:mem=30GB',
    project='P04010022',
    walltime='24:00:00',
    interface='mgt')

# scale up
cluster.scale(24)

# change your urls to the dask dashboard so that you can see it
dask.config.set({'distributed.dashboard.link':'https://jupyterhub.hpc.ucar.edu/stable/user/{USER}/proxy/{port}/status'})

# Setup your client
client = Client(cluster)

In [ ]:
#cluster.close()

In [ ]:
cluster

In [ ]:
h5a = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/*.h5a.*"))
h5i = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/*.h5i.*"))

In [ ]:
h5a = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/*.h5a.*"))
h5i = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/*.h5i.*"))

h5a = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/*.h5a.*"))
h5i = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/*.h5i.*"))

for ifile in np.arange(0,len(h5a),1):
#for ifile in np.arange(0,1,1):
    
    print(h5a[ifile])
    print(h5i[ifile])
    fname = os.path.basename(h5i[ifile])
    print(fname)

    # Extract timestamp: '2000-01-11' from '2000-01-11-03600'
    timestamp = fname.split(".")[-2]       # '2000-01-11-03600'
    date_str = timestamp.rsplit("-", 1)[0] # remove last '-03600'

    print(timestamp)
    # Build your new filename
    new_fname = f"CAM7_L58_HR_hourly.{date_str}.nc"
    print("output:", new_fname)

In [ ]:
h5a = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/more_files/*.h5a.*"))
h5i = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/more_files/*.h5i.*"))

h5a = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/*.h5a.*"))
h5i = sorted(glob.glob("/glade/campaign/cesm/community/amwg/cam_hr/f.e30_cam6_4_116.FHISTC_LTso.ne120_L58.c6aero.001/atm/hist/testing_MOAAP_prep/*.h5i.*"))


In [ ]:
%%time
# wmodified to add Tb

# Constants for Tb calculation
sigma = 5.67e-8        # W m-2 K-4
a = 1.228
b = -1.106e-3          # K-1

h5avars=['PRECT']

outdir = "/glade/derecho/scratch/hannay/MOAPP"
os.makedirs(outdir, exist_ok=True)

for ifile in np.arange(0,len(h5a),1):
#for ifile in np.arange(0,1,1):
    print("=====")
    print(h5a[ifile])
    print(h5i[ifile])
    fname = os.path.basename(h5i[ifile])
    #print(fname)
    # Extract timestamp: '2000-01-11' from '2000-01-11-03600'
    timestamp = fname.split(".")[-2]       # '2000-01-11-03600'
    date_str = timestamp.rsplit("-", 1)[0] # remove last '-03600'
    # Build your new filename
    new_fname = f"CAM7_L58_HR_hourly.{date_str}.nc"
    #print("output:", new_fname)
    
    # Full output path
    outpath = os.path.join(outdir, "MERGED_" + new_fname)

    # 👉 Skip if it already exists
    if os.path.exists(outpath):
        print(f"File already exists, skipping: {outpath}")
        continue

    print(f"Creating: {outpath}")
    # --- Open h5a and convert PRECT ---
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time":24}, engine="netcdf4")
    h5adat = h5adat[h5avars]
    prect_attrs = h5adat['PRECT'].attrs   # Save original metadata
    h5adat['PRECT'] = h5adat['PRECT'] * 1000 * 3600  # mm/hr  # Convert units
    h5adat['PRECT'].attrs = prect_attrs  # Restore metadata
    h5adat['PRECT'].attrs["units"] = "mm hr-1"     # Update units
    
    # --- Open h5i and compute Tb ---
    h5idat = xr.open_dataset(h5i[ifile], chunks={"time":24}, engine="netcdf4")
    FLUT = h5idat['FLUT']

    Tf = (FLUT / sigma) ** 0.25
    Tb = (-a + np.sqrt(a**2 + 4.0 * b * Tf)) / (2.0 * b)
    # Give Tb a name & metadata
    Tb = Tb.rename('Tb')
    Tb.attrs['long_name'] = "flux equivalent brightness temperature"
    Tb.attrs['units'] = "K"
    # Add Tb into h5idat so it will be included in the merge
    h5idat['Tb'] = Tb
    
    #!!! just putting the h5i time axis into h5a
    h5adat['time'] = h5idat.time.values

    newdat = xr.merge([h5adat, h5idat])
    newdat.to_netcdf(outpath, format='NETCDF4')
    newdat.close()
    h5adat.close()
    h5idat.close()

# ====== STOP ==== 

In [ ]:
%%time
# wmodified to add Tb

# Constants for Tb calculation
sigma = 5.67e-8        # W m-2 K-4
a = 1.228
b = -1.106e-3          # K-1

h5avars=['PRECT']

#for ifile in np.arange(0,len(h5a),1):
for ifile in np.arange(0,1,1):
    
    print(h5a[ifile])
    print(h5i[ifile])
    fname = os.path.basename(h5i[ifile])
    print(fname)
    # Extract timestamp: '2000-01-11' from '2000-01-11-03600'
    timestamp = fname.split(".")[-2]       # '2000-01-11-03600'
    date_str = timestamp.rsplit("-", 1)[0] # remove last '-03600'
    # Build your new filename
    new_fname = f"CAM7_L58_HR_hourly.{date_str}.nc"
    print("output:", new_fname)
    
    # --- Open h5a and convert PRECT ---
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time":24}, engine="netcdf4")
    h5adat = h5adat[h5avars]
    prect_attrs = h5adat['PRECT'].attrs   # Save original metadata
    h5adat['PRECT'] = h5adat['PRECT'] * 1000 * 3600  # mm/hr  # Convert units
    h5adat['PRECT'].attrs = prect_attrs  # Restore metadata
    h5adat['PRECT'].attrs["units"] = "mm hr-1"     # Update units
    
    # --- Open h5i and compute Tb ---
    h5idat = xr.open_dataset(h5i[ifile], chunks={"time":24}, engine="netcdf4")
    FLUT = h5idat['FLUT']

    Tf = (FLUT / sigma) ** 0.25
    Tb = (-a + np.sqrt(a**2 + 4.0 * b * Tf)) / (2.0 * b)
    # Give Tb a name & metadata
    Tb = Tb.rename('Tb')
    Tb.attrs['long_name'] = "flux equivalent brightness temperature"
    Tb.attrs['units'] = "K"
    # Add Tb into h5idat so it will be included in the merge
    h5idat['Tb'] = Tb
    
    #!!! just putting the h5i time axis into h5a
    h5adat['time'] = h5idat.time.values

    newdat = xr.merge([h5adat, h5idat])
    newdat.to_netcdf("/glade/derecho/scratch/hannay/MOAPP/MERGED_"+new_fname, format='NETCDF4')

In [ ]:
# works 
%%time
h5avars=['PRECT']
#for ifile in np.arange(0,len(h5a),1):
for ifile in np.arange(0,1,1):
    print(h5a[ifile])
    print(h5i[ifile])
    fname = os.path.basename(h5i[ifile])
    print(fname)
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time":24}, engine="netcdf4")
    h5adat = h5adat[h5avars]
   
    prect_attrs = h5adat['PRECT'].attrs   # Save original metadata
    h5adat['PRECT'] = h5adat['PRECT'] * 1000 * 3600  # mm/hr  # Convert units
    h5adat['PRECT'].attrs = prect_attrs  # Restore metadata
    h5adat['PRECT'].attrs["units"] = "mm hr-1"     # Update units

    h5idat = xr.open_dataset(h5i[ifile], chunks={"time":24}, engine="netcdf4")

    #!!! just putting the h5i time axis into h5a
    h5adat['time'] = h5idat.time.values

    newdat = xr.merge([h5adat, h5idat])
    newdat.to_netcdf("/glade/derecho/scratch/hannay/MOAPP/MERGED_"+fname+".nc", format='NETCDF4')

In [ ]:
# orig from Isla

%%time
h5avars=['PRECT']
#for ifile in np.arange(0,len(h5a),1):
for ifile in np.arange(0,1,1):
    print(h5a[ifile])
    print(h5i[ifile])
    fname = os.path.basename(h5i[ifile])
    print(fname)
    h5adat = xr.open_dataset(h5a[ifile], chunks={"time":24}, engine="netcdf4")
    h5adat = h5adat[h5avars]
    h5adat['PRECT'] = h5adat.PRECT*1000*3600 # ; unit: mm/hr

    h5idat = xr.open_dataset(h5i[ifile], chunks={"time":24}, engine="netcdf4")

    #!!! just putting the h5i time axis into h5a
    h5adat['time'] = h5idat.time.values

    newdat = xr.merge([h5adat, h5idat])
    newdat.to_netcdf("/glade/derecho/scratch/hannay/MOAPP/MERGED_"+fname+".nc", format='NETCDF4')

In [ ]:
newdat = xr.merge([h5adat, h5idat])

In [ ]:
fname = os.path.basename(h5i[ifile])
print(fname)